# Оценка OCR до LLM обработки

Анализ сырых результатов парсинга: layout-боксы, разметка блоков, extracted текст и таблицы из `data/01_extracted_pages/`.

In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown, HTML
from IPython.display import Image as IPyImage
import pandas as pd

# Автоопределение: Docker (/workspace) или локально
if Path('/workspace/data').exists():
    BASE = Path('/workspace')
else:
    BASE = Path.cwd().parent.parent  # notebooks/eval/ -> корень проекта

OCR_DIR = BASE / 'data' / '01_extracted_pages'
print(f'BASE: {BASE}')
print(f'OCR_DIR exists: {OCR_DIR.exists()}')

# Список документов
docs = sorted([d for d in OCR_DIR.iterdir() if d.is_dir()])
print('Документы в 01_extracted_pages:')
for d in docs:
    pages = sorted(d.glob('page_*.json'))
    imgs  = sorted(d.glob('page_*.png'))
    print(f'  {d.name}: {len(pages)} страниц, {len(imgs)} PNG')

In [ ]:
# --- Выберите документ ---
DOC = docs[0]  # или docs[1], docs[2]...

# Загружаем все страницы
pages = []
for p in sorted(DOC.glob('page_*.json')):
    try:
        data = json.loads(p.read_text(encoding='utf-8'))
        pages.append(data)
    except Exception as e:
        print(f'Ошибка {p.name}: {e}')

print(f'Документ: {DOC.name}')
print(f'Страниц:  {len(pages)}')

# Подсчёт таблиц (обработка списков результатов)
n_tables = 0
for p in pages:
    parsed_items = p if isinstance(p, list) else [p]
    for item in parsed_items:
        res = item.get('res', item) if isinstance(item, dict) else item
        if isinstance(res, dict):
            n_tables += len(res.get('table_res_list', []) or [])

print(f'Всего таблиц: {n_tables}')

## Layout-боксы и разметка блоков (PP-StructureV3)

In [ ]:
from PIL import Image as PILImage, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re

LABEL_COLORS = {
    'text':           '#2196F3',
    'paragraph_title':'#E91E63',
    'title':          '#E91E63',
    'figure':         '#FF9800',
    'image':          '#FF9800',
    'figure_caption': '#FF5722',
    'figure_title':   '#FF5722',
    'table':          '#4CAF50',
    'aside_text':     '#9C27B0',
    'header':         '#9E9E9E',
    'footer':         '#9E9E9E',
    'page_number':    '#9E9E9E',
    'number':         '#9E9E9E',
}
DEFAULT_COLOR = '#607D8B'

def draw_ocr_boxes(png_path, json_path, min_score):
    """Отрисовка layout-боксов на странице."""
    img = PILImage.open(png_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    data = json.loads(json_path.read_text(encoding='utf-8'))
    res = data[0]['res'] if isinstance(data, list) else data.get('res', data)

    used_labels = set()
    drawn_count = 0
    for box in res.get('layout_det_res', {}).get('boxes', []):
        coord = box.get('coordinate')
        label = box.get('label', '?')
        score = box.get('score', 1.0)
        if not coord or len(coord) != 4 or score < min_score:
            continue
        x1, y1, x2, y2 = coord
        color = LABEL_COLORS.get(label, DEFAULT_COLOR)
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1 + 2, max(0, y1 - 14)), f'{label} {score:.2f}', fill=color)
        used_labels.add(label)
        drawn_count += 1

    patches = [mpatches.Patch(color=LABEL_COLORS.get(l, DEFAULT_COLOR), label=l)
               for l in sorted(used_labels)]
    fig, ax = plt.subplots(figsize=(10, 14))
    ax.imshow(img); ax.axis('off')
    if patches:
        ax.legend(handles=patches, loc='upper right', fontsize=8,
                  framealpha=0.8, bbox_to_anchor=(1.18, 1))
    plt.title(f'OCR Boxes: {drawn_count} элементов')
    plt.tight_layout()
    plt.show()

# ── Настройки ──────────────────────────────────────────────
PAGE_FROM  = 1
PAGE_TO    = 5
MIN_SCORE  = 0.3
# ───────────────────────────────────────────────────────────

In [ ]:
# Функции для извлечения текста
def html_to_md(html):
    """Конвертация HTML таблицы в Markdown."""
    rows = re.findall(r'<tr>(.*?)</tr>', html, re.DOTALL)
    if not rows:
        return html
    out = []
    for i, row in enumerate(rows):
        cells = re.findall(r'<t[dh][^>]*>(.*?)</t[dh]>', row, re.DOTALL)
        cells = [re.sub(r'<[^>]+>', '', c).strip() for c in cells]
        out.append('| ' + ' | '.join(cells) + ' |')
        if i == 0:
            out.append('|' + '|'.join(['---'] * len(cells)) + '|')
    return '\n'.join(out)

SKIP_LABELS = {'header', 'footer', 'number', 'page_number', 'aside_text'}
FIGURE_LABELS  = {'figure', 'image'}
CAPTION_LABELS = {'figure_caption', 'figure_title'}

def inside_figure(bbox, fig_bboxes, thresh=0.6):
    if not bbox or len(bbox) < 4:
        return False
    bx1, by1, bx2, by2 = bbox
    b_area = max(0, bx2 - bx1) * max(0, by2 - by1)
    if b_area == 0:
        return False
    for fb in fig_bboxes:
        if not fb or len(fb) < 4:
            continue
        fx1, fy1, fx2, fy2 = fb
        ix = max(0, min(bx2, fx2) - max(bx1, fx1))
        iy = max(0, min(by2, fy2) - max(by1, fy1))
        if ix * iy / b_area >= thresh:
            return True
    return False

def extract_content(parsed_items, show_labels=False):
    """Извлечение текста, таблиц и информации о рисунках."""
    blocks, tables, fig_bboxes, fig_captions = [], [], [], []

    for item in parsed_items:
        res = item.get('res', item) if isinstance(item, dict) else item
        if not isinstance(res, dict):
            continue

        for t in res.get('table_res_list', []) or []:
            html = t.get('pred_html') or t.get('html')
            if html:
                tables.append(html)

        parsing = res.get('parsing_res_list') or []

        for b in parsing:
            if b.get('block_label') in FIGURE_LABELS:
                bb = b.get('block_bbox')
                if bb and len(bb) >= 4:
                    fig_bboxes.append(bb)
            if b.get('block_label') in CAPTION_LABELS:
                txt = (b.get('block_content') or '').strip()
                if txt:
                    fig_captions.append(txt)

        for b in parsing:
            label = b.get('block_label', '')
            if label in FIGURE_LABELS or label in CAPTION_LABELS:
                continue
            content = (b.get('block_content') or '').strip()
            if not content or label in SKIP_LABELS:
                continue
            bbox = b.get('block_bbox') or [0, 0, 0, 0]
            if inside_figure(bbox, fig_bboxes):
                continue
            if 'title' in label:
                content = f'## {content}'
            order = b.get('block_order')
            blocks.append({
                'order': order if order is not None else 9999,
                'y': bbox[1] if len(bbox) >= 2 else 0,
                'text': content,
                'label': label if show_labels else None,
            })

    blocks.sort(key=lambda x: (x['order'] or 9999, x['y'] or 0))
    # Разделение блоков в зависимости от типа
    text_parts = []
    for b in blocks:
        text_parts.append(b['text'])
    text = '\n\n'.join(text_parts).strip()
    tables_md = [html_to_md(t) for t in tables]
    return text, tables_md, fig_bboxes, fig_captions

# ──────────────────────────────────────────────────────────
# Просмотр: сначала layout-боксы на картинке, потом текст
SHOW_TEXT = True
SHOW_FIGS = True
SHOW_TBLS = True
SHOW_LABELS = False  # Показывать типы блоков (text, title, paragraph_title и т.д.)

for page_num in range(PAGE_FROM, PAGE_TO + 1):
    png_path  = DOC / f'page_{page_num:03d}.png'
    json_path = DOC / f'page_{page_num:03d}.json'

    if not png_path.exists() or not json_path.exists():
        print(f'Страница {page_num}: файлы не найдены')
        continue

    display(Markdown(f'---\n# Страница {page_num}'))
    
    # 1️⃣ Сначала - layout-боксы на картинке
    draw_ocr_boxes(png_path, json_path, MIN_SCORE)
    
    # 2️⃣ Потом - извлеченный текст и таблицы
    parsed = json.loads(json_path.read_text(encoding='utf-8'))
    parsed_items = parsed if isinstance(parsed, list) else [parsed]
    text, tables_md, fig_bboxes, fig_captions = extract_content(parsed_items, show_labels=SHOW_LABELS)

    if SHOW_TEXT:
        display(Markdown(f'**Текст ({len(text)} символов):**\n\n{text}' if text else '*Текст пуст*'))

    if SHOW_FIGS:
        figs = sorted(DOC.glob(f'page_{page_num:03d}_fig*.png'))
        if figs:
            display(Markdown(f'**Рисунки ({len(figs)} шт.):**'))
            for i, f in enumerate(figs):
                cap = fig_captions[i] if i < len(fig_captions) else ''
                if cap:
                    display(Markdown(f'*{cap}*'))
                display(IPyImage(filename=str(f), width=500))

    if SHOW_TBLS and tables_md:
        display(Markdown(f'**Таблицы ({len(tables_md)} шт.):**'))
        for i, t in enumerate(tables_md):
            display(Markdown(f'*Таблица {i+1}:*\n\n{t}'))